# MachineInnovators — review del progetto MLOps

Repository: [social-sentiment-mlops](https://github.com/flaviotorre-dev/social-sentiment-mlops)  
Immagine pubblica: [GitHub Container Registry](https://github.com/flaviotorre-dev/social-sentiment-mlops/pkgs/container/social-sentiment-mlops)

Progetto didattico: espone un classificatore pre-addestrato per sentiment positive, neutral e negative. Questo notebook documenta le scelte e i limiti; non esegue training.

## Architettura

```text
Client --HTTP--> FastAPI (/predict, /health)
                     | modello caricato all'avvio
                     v
       CardiffNLP Twitter RoBERTa sentiment

Prometheus --scrapes /metrics--> FastAPI
Grafana --queries-------------> Prometheus

GitHub Actions --test/build--> GHCR
Airflow DAG settimanale: collect_data -> evaluate_model -> decide -> retrain | skip
```

L'API, Prometheus e Grafana sono avviabili con Docker Compose. Il DAG è un file dimostrativo e non richiede che Airflow faccia parte dello stack applicativo.

## Perché queste scelte

- **FastAPI**: API HTTP piccola, validazione della richiesta e documentazione OpenAPI automatica.
- **Modello pre-addestrato CardiffNLP**: evita di inventare un dataset o un training per coprire il caso d'uso didattico.
- **Python 3.14**: versione fissata e usata coerentemente in ambiente, Docker e CI.
- **PyTorch CPU-only**: non serve una GPU per il demo e si evita la dipendenza CUDA; l'immagine resta comunque grande (circa 3,82 GB).
- **Due job CI**: il job di build dipende dal successo di lint e test; non si pubblica un'immagine se i controlli falliscono.
- **GHCR**: la pubblicazione usa il `GITHUB_TOKEN` e non richiede un token di registry aggiuntivo.

Alternative non adottate: un modello distillato o ONNX per ridurre dimensioni e latenza; una build Docker multi-stage; un deploy su Hugging Face Spaces. Quest'ultimo richiederebbe uno Space Docker, un `HF_TOKEN` conservato nei secret del repository e un passaggio di deploy dedicato.

## CI/CD

La GitHub Action esegue `ruff check`, `ruff format --check` e pytest con Python 3.14 e cache del modello Hugging Face. Se il job `test` riesce, un push su `main` costruisce l'immagine e la pubblica su GHCR con tag `latest` e tag SHA. Pull request e push eseguono i controlli; il job di pubblicazione è riservato ai push su `main`.

La pipeline verifica qualità e pubblicazione dell'immagine, ma non effettua il deploy dell'applicazione su un ambiente di produzione.

## Monitoraggio

Prometheus raccoglie `/metrics` ogni 15 secondi; Grafana carica datasource e dashboard tramite provisioning. I segnali sono:

- volume di richieste per endpoint;
- latenza, con visualizzazione del p95;
- richieste con status 4xx/5xx;
- distribuzione delle classi predette;
- lunghezza dei testi in caratteri.

L'unico alert didattico è **quota di predizioni negative > 60%** sulle predizioni degli ultimi 15 minuti, mantenuta per 5 minuti. È una soglia semplice per segnalare uno spostamento degli output, non una soglia calibrata: per un servizio reale andrebbe definita con dati storici e tolleranza ai cambiamenti normali di traffico. L'alert è visibile in Prometheus, ma non sono configurate notifiche esterne.

## Data drift e concept drift

**Data drift** significa che cambia la distribuzione degli input, ad esempio lessico, lunghezza o argomenti dei testi. Può essere osservato anche senza conoscere il sentiment corretto, confrontando le caratteristiche degli input nel tempo.

**Concept drift** significa che cambia la relazione tra testo e sentiment (l'etichetta corretta). La sola distribuzione degli input o delle predizioni non basta a confermarlo: servono etichette reali su un campione di testi e una misura periodica degli errori del modello. Le metriche attuali descrivono traffico e output, non misurano l'accuracy online.

## Logica del DAG

Il DAG `weekly_sentiment_retraining` è pianificato con `@weekly`:

1. `collect_data` restituisce un conteggio dimostrativo dei campioni via XCom.
2. `evaluate_model` legge quel valore da XCom e restituisce un'accuracy stub.
3. `decide` legge l'accuracy e confronta con la soglia 0,75.
4. Se l'accuracy è sotto soglia, il branch esegue `retrain`; altrimenti `skip`.

Nel codice demo l'accuracy stub è 0,72, quindi il percorso scelto è `retrain`. Nessun dato viene realmente raccolto, nessun modello viene valutato e il task `retrain` non addestra un modello.

## Limiti attuali

- Il modello non è stato fine-tuned sui dati di un cliente e le classi non sono una misura di verità.
- Non ci sono autenticazione, rate limiting, gestione di dati sensibili o obiettivi SLO.
- Il servizio carica un solo modello in memoria e non è dimensionato per alta disponibilità.
- Il conteggio per classe non permette di calcolare accuracy senza etichette reali.
- L'immagine Docker è pesante; il compose è per una demo locale, senza backup, TLS o notifiche degli alert.
- I task Airflow sono stub e non esiste una pipeline di training o validazione reale.

## Cosa servirebbe per un cliente reale

Prima del rilascio servirebbero requisiti e dataset rappresentativi, valutazione per segmento e lingua, revisione di bias e privacy, autorizzazione e protezione degli endpoint, logging conforme alle policy, deployment con health check e rollback, risorse dimensionate, alert con un canale di notifica, metriche di qualità basate su etichette e un processo di approvazione per ogni nuovo modello. I dati degli utenti andrebbero minimizzati e conservati secondo una policy concordata.

## Cosa farei con più tempo

Misurerei latenza e memoria con test di carico, confronterei il modello attuale con una variante più piccola, fisserei e verificherei tutte le immagini di base tramite digest, aggiungerei test di integrazione del compose e controlli sul formato delle metriche, e definirei una valutazione offline con campioni etichettati. Solo dopo aver stabilito requisiti di qualità e operativi aggiungerei retraining reale, model registry e deploy controllato.